# 05 · 메모리 관리 & 성능측정

> **CuPy 2일 집중 코스 — Day 1 / 단원 3 (메모리 관리 및 성능측정, 1.5H)**

GPU 성능의 절반은 **데이터 이동**과 **메모리 사용**에서 결정됩니다. 암묵적 전송을 피하고,
메모리풀과 임시버퍼를 다루며, 프로파일링으로 병목을 찾는 법을 배웁니다.

## 학습 목표
- host/device 메모리 공간과 전송(`asarray`/`asnumpy`)을 이해하고 **데이터를 GPU에서 생성**한다.
- **암묵적 전송·동기화**를 유발하는 연산을 식별하고 피한다.
- **메모리풀**을 관찰하고, `out=`으로 임시버퍼(temporary)를 줄인다.
- `benchmark`/`time_range`/`profile` 로 병목을 측정·표시한다.

## 목차
1. [host/device 메모리 공간](#1)
2. [암묵적 전송 주의](#2)
3. [메모리풀 관찰](#3)
4. [임시버퍼 줄이기 (out=)](#4)
5. [전송 병목 비교](#5)
6. [프로파일링 도구](#6)
7. [연습문제](#7)
8. [(실전) Power Iteration](#8)
9. [체크포인트](#9)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare, bytes_human
print_env()

<a id="1"></a>
## 1. host/device 메모리 공간

이기종(heterogeneous) 시스템은 **두 메모리 공간**으로 나뉩니다: CPU가 접근하는 **Host Memory**, GPU가 접근하는 **Device Memory**.
연산하려면 데이터가 해당 프로세서의 메모리에 있어야 하므로 **명시적 전송**이 필요합니다.
- Host → Device: `x_dev = cp.asarray(x_host)`
- Device → Host: `y_host = cp.asnumpy(y_dev)`

<img src="images/figures/new_host_device_transfer.png" width="640">



**전송보다 생성이 싸다**: 큰 데이터를 host에서 만들어 옮기는 것보다 **GPU에서 바로 생성**하는 편이 보통 빠릅니다.

In [ ]:
N = 4096
# (A) host 생성 후 전송
def make_transfer():
    a = np.random.random((N, N)).astype(np.float32)
    return cp.asarray(a)            # host->device 전송 포함

# (B) GPU에서 직접 생성
def make_on_gpu():
    return cp.random.random((N, N), dtype=cp.float32)

compare('make', make_transfer, make_on_gpu, n_repeat=10, n_warmup=2)
print('=> 같은 데이터라도 GPU 직접 생성이 전송보다 보통 빠릅니다.')

<a id="2"></a>
## 2. 암묵적 전송 주의

CuPy는 다음 상황에서 **조용히 전송·동기화**하여 성능을 갉아먹습니다.
- **출력/표현**: `print(x)`, `str(x)`, f-string 보간
- **파이썬 스칼라/리스트 변환**: `int(x)`, `float(x)`, `bool(x)`, `x.item()`
- **GPU 버전이 없어 NumPy/SciPy로 폴백**하는 함수

또한 `CuPy + NumPy`(rank≥1) 연산은 **에러**입니다(같은 장치에 있어야 함). 0-차원(스칼라) 배열은 일부 예외가 있습니다.

<img src="images/figures/new_implicit_transfers.png" width="620">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
n = 1_000_000 # n 을 변경해보세요 e.g.) n = 5_000_000
n_steps = 30 # n_steps 를 변경해보세요 e.g.) n_steps = 500 
x = cp.random.random(n, dtype=cp.float32)

# (나쁨) 루프마다 float()로 스칼라 변환 -> 매번 동기화/전송
def with_sync(x, steps=n_steps):
    acc = 0.0
    for _ in range(steps):
        acc += float((cp.sin(x) + 1).sum())   # float() = 암묵적 전송+동기화
    return acc

# (좋음) GPU에 누적해 두고 마지막에 한 번만 전송
def without_sync(x, steps=n_steps):
    acc = cp.zeros((), dtype=cp.float64)
    for _ in range(steps):
        acc += (cp.sin(x) + 1).sum()
    return float(acc)                          # 마지막 한 번만 전송

compare('sync-in-loop', lambda: with_sync(x), lambda: without_sync(x), n_repeat=5, n_warmup=1)

<a id="3"></a>
## 3. 메모리풀 관찰

CuPy는 `cudaMalloc/Free` 비용을 줄이려 **메모리풀**을 사용합니다(기본 활성). 해제한 배열의 메모리는 풀에 남아 재사용됩니다.
- `cp.get_default_memory_pool()` → `used_bytes()`(사용 중), `total_bytes()`(풀이 보유), `free_all_blocks()`(미사용 블록 반환)

In [ ]:
mempool = cp.get_default_memory_pool()
a = cp.random.random((4096, 4096), dtype=cp.float32)   # ~64MB 할당
print('할당 후  used :', bytes_human(mempool.used_bytes()), '| total:', bytes_human(mempool.total_bytes()))
del a                                                   # 파이썬 참조 해제(풀에는 남음)
print('del 후   used :', bytes_human(mempool.used_bytes()), '| total:', bytes_human(mempool.total_bytes()))
mempool.free_all_blocks()                               # 풀의 미사용 블록 OS 반환
print('free 후  used :', bytes_human(mempool.used_bytes()), '| total:', bytes_human(mempool.total_bytes()))

<a id="4"></a>
## 4. 임시버퍼(temporary) 줄이기 — `out=`

`b = cp.sin(a); c = cp.cos(a); d = b*b + c*c` 처럼 식이 길면 **중간 배열(temporary)** 이 여러 개 생겨 메모리·대역폭을 낭비합니다.
미리 버퍼를 잡고 `out=`에 써 넣으면 할당을 줄일 수 있습니다.

In [ ]:
a = cp.random.random(20_000_000, dtype=cp.float32)

def slow_step(a):
    b = cp.sin(a); c = cp.cos(a); d = b*b + c*c
    return cp.sqrt(d)

def fast_step(a, b, c, d):
    cp.sin(a, out=b); cp.cos(a, out=c)
    cp.multiply(b, b, out=d); d += c*c
    cp.sqrt(d, out=d); return d

b = cp.empty_like(a); c = cp.empty_like(a); d = cp.empty_like(a)
r_slow = bench(lambda: slow_step(a), n_repeat=10, name='temporary 많음')
r_fast = bench(lambda: fast_step(a, b, c, d), n_repeat=10, name='out= 재사용')
print_bench(r_slow); print_bench(r_fast)
print(f'speedup: {gpu_ms(r_slow)/gpu_ms(r_fast):.2f}x')

<a id="5"></a>
## 5. 전송 병목 비교

루프 안에서 반복적으로 host로 가져오면(`asnumpy`/`float`) 전송이 누적되어 병목이 됩니다.
2절과 같은 원리지만, 여기서는 더 큰 데이터로 전송 비용의 크기를 봅니다.

In [ ]:
x = cp.random.random(10_000_000, dtype=cp.float32)

def heavy(x, steps=20):   # 매 스텝 host 전송
    acc = 0.0
    for _ in range(steps):
        acc += float((cp.sin(x) + 1).sum())
    return acc

def light(x, steps=20):   # GPU 누적, 마지막에 1회 전송
    acc = cp.zeros((), dtype=cp.float64)
    for _ in range(steps):
        acc += (cp.sin(x) + 1).sum()
    return float(acc)

compare('transfer', lambda: heavy(x), lambda: light(x), n_repeat=3, n_warmup=1)

<a id="6"></a>
## 6. 프로파일링 도구

성능 개선은 **측정 → 가설 → 수정** 순서로 합니다. CuPy가 제공하는 도구:
- **`cupyx.profiler.benchmark`** (= `course_utils.bench`): 워밍업·동기화·반복평균 자동.
- **`cupyx.profiler.time_range`**: 코드 구간을 **NVTX 범위**로 표시(데코레이터/컨텍스트 매니저). Nsight Systems 타임라인에 나타남.
- **`cupyx.profiler.profile`**: 프로파일러 캡처 구간을 켜고 끄는 컨텍스트 매니저. `nsys --capture-range=cudaProfilerApi` 와 함께 사용.

Jupyter notebook console에서 다음을 수행합니다.

1. 최신 Ubuntu 24.04용 cuda-keyring 패키지 다운로드

In [ ]:
! wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/cuda-keyring_1.1-1_all.deb -O cuda-keyring.deb 

2. dpkg 명령어로 keyring 설치 (NVIDIA 저장소 인증 키 등록)

In [ ]:
! dpkg -i cuda-keyring.deb

3. 사용한 임시 deb 파일 삭제

In [ ]:
! rm cuda-keyring.deb

4. NVIDIA 신규 저장소가 반영되도록 apt 업데이트

In [ ]:
! apt-get update

5. CUDA 버전별 nsight-systems 패키지 설치

In [ ]:
! apt-get install -y cuda-nsight-systems-12-6 || apt-get install -y cuda-nsight-systems-12-5 || apt-get install -y cuda-nsight-systems-12-8

6. nsys 설치 확인

In [ ]:
! nsys --version

7. 4에서 저장소 에러나올 경우 아래 커맨드로 [signed-by=...] 이 없는 중복 파일 확인 후 삭제

In [ ]:
! grep -RIn "developer.download.nvidia.com/compute/cuda" /etc/apt/sources.list /etc/apt/sources.list.d/ 2>/dev/null

`!nsys profile python ./cupy/05_6_profiling_test.py` 실행후, 출력파일(확장자 nsys-rep) 다운로드 하신뒤, 로컬 컴퓨터에서 nsight 프로그램을 실행해서 타임라인을 확인해보세요. `time_range`로 구간을 라벨링하세요.

In [ ]:
%%writefile 05_6_profiling_test.py
import cupy as cp
import time
from cupyx.profiler import time_range
from cupy.cuda import profiler

print("--- NVTX 프로파일링 테스트 시작 ---")

# 1.  데이터 준비
x = cp.random.random(20_000_000, dtype=cp.float32)

# 2. nsys 프로파일러 수집 시작 명령 
profiler.start()

# [테스트 1] 컨텍스트 매니저 방식
print("Step 1: Context Manager 테스트 중...")
with time_range('MY_CONTEXT_RANGE', color_id=1):
    for _ in range(20):
        y = (cp.sin(x) + 1).sum()
    cp.cuda.Device().synchronize() # NVTX 영역이 닫히기 전 GPU 연산 보장

time.sleep(0.5) # 타임라인 구분용 공백 시간

# [테스트 2] 데코레이터 방식
@time_range('MY_DECORATOR_RANGE', color_id=3)
def run_stage(data):
    for _ in range(20):
        data = (cp.cos(data) ** 2).sum()
    return data.sum()

print("Step 2: Decorator 테스트 중...")
res = run_stage(x)
cp.cuda.Device().synchronize()

# 3. 프로파일러 수집 종료 명령
profiler.stop()

print("--- 테스트 완료! 결과를 nsys-rep 파일로 저장합니다. ---")


In [ ]:
! nsys profile --force-overwrite true -o profiling_test python 05_6_profiling_test.py

### 6.1 CUB/cuTENSOR 백엔드 — "공짜" 가속

리덕션(`sum`, `prod`, `min/max`, `argmin/argmax`, `cumsum` 등)은 **CUB**/**cuTENSOR** 백엔드로 더 빨라질 수 있습니다.
환경변수 **`CUPY_ACCELERATORS`** 로 선택합니다(시도 순서대로). 예: 세션을 `CUPY_ACCELERATORS=cub,cutensor python` 으로 시작.

- CuPy **v11+ 는 기본으로 CUB를 사용**합니다(끄려면 `CUPY_ACCELERATORS=""`).
- cuTENSOR는 별도 설치 시 이항 ufunc·리덕션·텐서 축약을 가속합니다.
- 환경변수는 **프로세스 시작 전에** 설정해야 하며, 데이터 레이아웃(연속 축 여부)에 따라 효과가 달라 — 항상 벤치마크로 확인하세요.

In [ ]:
%%writefile cubtest.py

import os
#os.environ["CUPY_ACCELERATORS"] = ""  
os.environ["CUPY_ACCELERATORS"] = "cub"  
import cupy as cp
from course_utils import bench, print_bench

# 현재 백엔드로 큰 배열 리덕션 측정 (CUPY_ACCELERATORS 설정에 따라 속도가 달라짐)
a = cp.random.random((256, 256, 256), dtype=cp.float32)
print_bench(bench(a.sum, n_repeat=50, n_warmup=5, name='sum (현재 백엔드)'))
# 비교 실험: 위의 environ 주석을 번갈아가면서 실행하고, '!python ./cupy/cubtest.py'를 각각 수행해보세요.  

<a id="7"></a>
## 7. 연습문제 — 비효율 파이프라인 2배 개선

아래 `pipeline_slow`는 (1) 데이터를 host에서 만들어 전송하고, (2) temporary가 많으며, (3) 루프마다 스칼라를 host로 가져옵니다.
세 가지를 고쳐 `pipeline_fast`를 만들고, `compare`로 **2배 이상** 빨라지는지 확인하세요.

In [ ]:
def pipeline_slow(N=10_000_000, steps=15):
    a = cp.asarray(np.random.random(N).astype(np.float32))   # (1) host 생성+전송
    total = 0.0
    for _ in range(steps):
        b = cp.sin(a); c = cp.cos(a); d = b*b + c*c           # (2) temporary 다수
        total += float(cp.sqrt(d).sum())                      # (3) 매 스텝 host 전송
    return total

def pipeline_fast(N=10_000_000, steps=15):
    # TODO: (1) cp.random로 GPU 직접 생성  (2) out= 버퍼 재사용  (3) GPU 누적 후 1회 전송
    raise NotImplementedError

# compare('pipeline', lambda: pipeline_slow(), lambda: pipeline_fast(), n_repeat=3, n_warmup=1)

<details>
<summary>💡 해답 보기</summary>

```python
def pipeline_fast(N=10_000_000, steps=15):
    a = cp.random.random(N, dtype=cp.float32)        # (1) GPU 직접 생성
    b = cp.empty_like(a); c = cp.empty_like(a); d = cp.empty_like(a)
    total = cp.zeros((), dtype=cp.float64)            # (3) GPU 누적
    for _ in range(steps):
        cp.sin(a, out=b); cp.cos(a, out=c)            # (2) out= 재사용
        cp.multiply(b, b, out=d); d += c*c
        cp.sqrt(d, out=d)
        total += d.sum()
    return float(total)                               # 마지막 1회만 전송

compare('pipeline', lambda: pipeline_slow(), lambda: pipeline_fast(), n_repeat=3, n_warmup=1)
```

세 가지 개선(생성·temporary·전송)이 합쳐져 보통 2배 이상 빨라집니다. 어떤 요인이 가장 컸는지 하나씩 켜고 꺼 보세요.
</details>

<a id="8"></a>
## 8. (실전) Power Iteration — 메모리 공간 적용

지금까지 배운 원칙(전송 최소화·GPU에서 생성·암묵적 동기화 회피)을 **현실적 알고리즘**에 적용합니다.
**거듭제곱 반복법(Power Iteration)** 은 행렬의 **최대 고유값**을 구합니다: `y = A x` → 정규화를 반복.
포팅은 `np.`→`cp.` 치환이며, 핵심은 **A를 GPU에서 직접 생성**하고 반복 중 host 전송을 피하는 것입니다.
(GTC Memory Spaces 예제를 본 과정 형식으로 재구성)

In [ ]:
def make_spd(xp, n):
    M = xp.random.random((n, n)).astype(xp.float32)
    return (M + M.T) / 2          # 대칭 -> 실수 고유값 보장

def power_iteration(xp, A, iters=300):
    x = xp.ones(A.shape[0], dtype=A.dtype)
    for _ in range(iters):
        y = A @ x
        x = y / xp.linalg.norm(y)   # 반복 중 host 전송 없음 (모두 GPU에 머무름)
    return float((x @ (A @ x)) / (x @ x))   # 마지막에 한 번만 스칼라 전송

n = 2000

# --- 실행 및 검증 ---
A_np = make_spd(np, n)
# [수정 사항] CPU 데이터를 전송하는 대신, GPU에서 직접 무작위 대칭 행렬을 바로 생성합니다.
A_cp = make_spd(cp, n)

lam_np = power_iteration(np, A_np)
lam_cp = power_iteration(cp, A_cp)
ref = float(np.linalg.eigvalsh(A_np).max())
print(f'CPU λ={lam_np:.5f} | GPU λ={lam_cp:.5f} | ref(eigvalsh)={ref:.5f}')


In [ ]:
# CPU vs GPU 속도 (같은 행렬 A_np 입력으로 공정 비교)
compare('power_iter', 
        lambda: power_iteration(np, A_np),
        lambda: power_iteration(cp, A_cp), n_repeat=3, n_warmup=1)
print('=> A를 GPU에서 직접 생성하면 host->device 전송까지 아낍니다 (make_spd(cp, n)).')

## 🧪 추가 연습 & 실험

**연습 — `out=`로 temporary 제거**: 아래 식을 `out=` 버퍼로 다시 써서 임시배열을 없애고 속도를 비교하세요.
식: `r = sqrt(exp(-a) + log1p(a))`

In [ ]:
a = cp.random.random(20_000_000, dtype=cp.float32)
def f_slow(a):
    return cp.sqrt(cp.exp(-a) + cp.log1p(a))
def f_fast(a, t1, t2):
    # TODO: cp.exp(-a,out=t1); cp.log1p(a,out=t2); t1+=t2; cp.sqrt(t1,out=t1); return t1
    raise NotImplementedError

t1 = cp.empty_like(a); t2 = cp.empty_like(a)
# r_slow=bench(lambda:f_slow(a),name='slow'); r_fast=bench(lambda:f_fast(a,t1,t2),name='fast')
# print_bench(r_slow); print_bench(r_fast); print('speedup', round(gpu_ms(r_slow)/gpu_ms(r_fast),2))

<details><summary>💡 해답 보기</summary>

```python
def f_fast(a, t1, t2):
    cp.exp(cp.negative(a), out=t1)   # 또는 cp.exp(-a, out=t1)
    cp.log1p(a, out=t2)
    t1 += t2
    cp.sqrt(t1, out=t1)
    return t1

t1 = cp.empty_like(a); t2 = cp.empty_like(a)
r_slow = bench(lambda: f_slow(a), name='slow')
r_fast = bench(lambda: f_fast(a, t1, t2), name='fast')
print_bench(r_slow); print_bench(r_fast)
print('speedup', round(gpu_ms(r_slow)/gpu_ms(r_fast), 2))
```
</details>

**실험 — 메모리풀 증가 관찰**: 크기를 키우며 `used/total`을 출력하고, `free_all_blocks()` 전후를 비교하세요.

In [ ]:
mp = cp.get_default_memory_pool()
for n in [1, 4, 16, 64]:
    x = cp.random.random(n*1_000_000, dtype=cp.float32)
    print(f'{n:>3}M elems | used {bytes_human(mp.used_bytes()):>10} | total {bytes_human(mp.total_bytes()):>10}')
    del x
mp.free_all_blocks()
print('free 후   | used', bytes_human(mp.used_bytes()), '| total', bytes_human(mp.total_bytes()))

<a id="8"></a>
## 9. 체크포인트

- [ ] host/device 전송과 'GPU에서 생성' 이점을 안다
- [ ] 암묵적 전송(print/float/.item/폴백)을 식별하고 피한다
- [ ] 메모리풀의 used/total/free_all_blocks를 관찰했다
- [ ] `out=`으로 temporary를 줄여 속도를 높였다
- [ ] `benchmark`/`time_range`/`profile`의 용도를 구분한다
- [ ] 연습: 파이프라인을 2배 이상 개선했다

다음: **`06_streams_async`** — 스트림·이벤트·비동기 전송으로 연산과 전송을 겹칩니다.